## 逐步拆解

```python
print(f"解释方差比: {np.sum(ica.pca_explained_variance_[:n_components]) / np.sum(ica.pca_explained_variance_):.1%}")
```

---

### 背景：ICA 前面有个 PCA

MNE 的 ICA 内部流程是两步：

```text
原始数据 (60 通道)
     ↓
  PCA 降维  → 保留前 k 个主成分，扔掉方差极小的噪声维度
     ↓
  ICA 在这 k 个主成分上分解
     ↓
  n_components 个独立成分
```

`ica.pca_explained_variance_` 是 PCA 阶段**所有主成分**各自的方差（一个长度 = k 的数组）。

---

### `ica.pca_explained_variance_` 长什么样

```text
成分 0 方差: 3.2e-10   ← 最大（大部分脑电信号）
成分 1 方差: 1.8e-10
成分 2 方差: 0.9e-10
...
成分 14 方差: 0.1e-10   ← 第 n_components 个
成分 15 方差: 0.05e-10
...
成分 59 方差: 1e-15     ← 微小（基本是噪声）
```

---

### 分子：`np.sum(ica.pca_explained_variance_[:n_components])`

```python
ica.pca_explained_variance_[:15]   # 取前 15 个成分的方差列表
np.sum(...)                         # 把它们加起来
```

含义：**ICA 将用到的前 15 个 PCA 成分，一共"承载"了多少方差**。

---

### 分母：`np.sum(ica.pca_explained_variance_)`

**全部 PCA 成分的总方差**（原始数据的总能量）。

---

### 除法：解释方差比

```
解释方差比 = 前 15 个成分的方差之和 / 全部成分的方差之和
```

---

### 直观理解

```text
原始数据 = 60 通道 EEG，对应 60 个 PCA 成分
           ├── 前 15 个成分：承载了原始信号 85% 的方差（信息）
           └── 后 45 个成分：承载了 15% 的方差（主要是噪声碎屑）
                                 ↑ 被 PCA 阶段丢弃

解释方差比 = 85%
```

这个数字告诉你：**你选了 15 个成分，它们保留了多少"信息量"**。

---

### 多少算合理？

| 解释方差比 | 含义 |
|-----------|------|
| > 95% | 成分数可能太多（噪声也被纳入了） |
| **80% - 95%** | ✅ 合理范围 |
| 70% - 80% | 勉强可用，但可能丢了些弱信号 |
| < 70% | 成分数太少，信息损失严重 |

---

### 一句话总结

**"你挑了 15 个 ICA 成分，它们背后对应的 PCA 主成分一共保留了原始 EEG 数据 X% 的信息量。"** 这个数字是用来验证 `n_components` 是否选得合适的诊断指标。